# Dashboard: Portfolio Highlights

Reads each industry's business-metric table and renders one polished
Matplotlib/Seaborn chart per vertical — a quick "here's what this demo
environment covers" walkthrough across life sciences, healthcare, pharma,
agriculture, financial services, and digital marketing. Requires the
corresponding generator + transformation notebooks to have run first —
see `notebooks/README.md` for the full list.

Requires `pandas`, `matplotlib`, and `seaborn` — installed by the next cell.

In [ ]:
!pip3 install -q pandas matplotlib seaborn

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()
plt.style.use("seaborn-v0_8-whitegrid")

## Life Sciences — Trial Pipeline Health by Phase

In [ ]:
trials_pd = spark.table("clinical_trials.trial_pipeline_health_summary_v1").orderBy(F.desc("trial_count")).toPandas()

norm = plt.Normalize(trials_pd["termination_rate_pct"].min(), trials_pd["termination_rate_pct"].max())
colors = plt.cm.Blues(norm(trials_pd["termination_rate_pct"]))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(trials_pd["phase"], trials_pd["trial_count"], color=colors)
ax.set_ylabel("trial_count")
ax.set_title("Life Sciences — Trial Count and Termination Rate by Phase")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
sm = plt.cm.ScalarMappable(cmap="Blues", norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=ax, label="termination_rate_pct")
plt.tight_layout()
plt.show()

## Healthcare — Readmission Rate by Diagnosis

In [ ]:
readmit_pd = (
    spark.table("medicare_claims.readmission_risk_summary_v1")
    .orderBy(F.desc("total_admissions"))
    .limit(15)
    .toPandas()
    .sort_values("total_admissions")
)

norm = plt.Normalize(readmit_pd["readmission_rate_pct"].min(), readmit_pd["readmission_rate_pct"].max())
colors = plt.cm.Reds(norm(readmit_pd["readmission_rate_pct"]))

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(readmit_pd["primary_diagnosis_code"], readmit_pd["total_admissions"], color=colors)
ax.set_xlabel("total_admissions")
ax.set_ylabel("ICD-9 diagnosis code")
ax.set_title("Healthcare — Top 15 Diagnoses by Admission Volume (colored by 30-day Readmission Rate)")
sm = plt.cm.ScalarMappable(cmap="Reds", norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=ax, label="readmission_rate_pct")
plt.tight_layout()
plt.show()

## Pharma — Top Drug/Reaction Safety Signals

In [ ]:
signals_pd = (
    spark.table("pharmacovigilance.drug_safety_signal_summary_v1")
    .orderBy(F.desc("report_count"))
    .limit(15)
    .toPandas()
)
signals_pd["drug_reaction"] = signals_pd["drug_name"] + " → " + signals_pd["reaction_term"]
signals_pd = signals_pd.sort_values("report_count")

norm = plt.Normalize(signals_pd["serious_rate_pct"].min(), signals_pd["serious_rate_pct"].max())
colors = plt.cm.OrRd(norm(signals_pd["serious_rate_pct"]))

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(signals_pd["drug_reaction"], signals_pd["report_count"], color=colors)
ax.set_xlabel("report_count")
ax.set_title("Pharma — Top 15 Drug/Reaction Safety Signals")
sm = plt.cm.ScalarMappable(cmap="OrRd", norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=ax, label="serious_rate_pct")
plt.tight_layout()
plt.show()

## Agriculture — Biggest Crop Yield Movers

In [ ]:
yield_pd = (
    spark.table("agriculture.crop_yield_trend_summary_v1")
    .filter("yield_change_pct IS NOT NULL")
    .withColumn("state_commodity", F.concat_ws(" — ", "state", "commodity"))
    .orderBy(F.desc(F.abs("yield_change_pct")))
    .limit(15)
    .toPandas()
    .sort_values("yield_change_pct")
)

norm = mcolors.TwoSlopeNorm(
    vmin=yield_pd["yield_change_pct"].min(), vcenter=0, vmax=yield_pd["yield_change_pct"].max()
)
colors = plt.cm.RdYlGn(norm(yield_pd["yield_change_pct"]))

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(yield_pd["state_commodity"], yield_pd["yield_change_pct"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("yield_change_pct")
ax.set_title("Agriculture — Biggest Yield Movers, 2012 to 2017 Census")
plt.tight_layout()
plt.show()

## Financial Services — Fraud Rate by Amount Bucket

In [ ]:
fraud_pd = (
    spark.table("payments.fraud_risk_summary_v1")
    .groupBy("amount_bucket")
    .agg(
        F.sum("transaction_count").alias("transaction_count"),
        F.sum("fraud_count").alias("fraud_count"),
    )
    .withColumn("fraud_rate_pct", F.round(F.col("fraud_count") * 100.0 / F.col("transaction_count"), 3))
    .toPandas()
)
bucket_order = ["0-10", "10-50", "50-100", "100-500", "500+"]
fraud_pd["amount_bucket"] = pd.Categorical(fraud_pd["amount_bucket"], categories=bucket_order, ordered=True)
fraud_pd = fraud_pd.sort_values("amount_bucket")

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=fraud_pd, x="amount_bucket", y="fraud_rate_pct", order=bucket_order, color=sns.color_palette("Blues")[3], ax=ax)
ax.set_title("Financial Services — Fraud Rate by Transaction Amount Bucket")
plt.tight_layout()
plt.show()

## Digital Marketing — Customer Segment Distribution

In [ ]:
segments_pd = (
    spark.table("retail.customer_rfm_segments_v1")
    .groupBy("segment")
    .agg(F.count("*").alias("customers"), F.round(F.avg("monetary"), 2).alias("avg_monetary"))
    .toPandas()
)

fig, ax = plt.subplots(figsize=(7, 6))
ax.pie(
    segments_pd["customers"], labels=segments_pd["segment"], autopct="%1.0f%%",
    colors=sns.color_palette("Set2"), startangle=90,
)
ax.set_title("Digital Marketing — Customer Segment Distribution (RFM)")
plt.tight_layout()
plt.show()